# Week 51, Day 1: Dijkstra's algorithm

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w051d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
from collections import deque

roads = {"home": {"bridge": 9, "lane": 2}, "lane": {"field": 2}, "field": {"bridge": 2}, "bridge": {}}

hops = {"home": 0}
q = deque(["home"])
while q:
    v = q.popleft()
    for w in roads[v]:
        if w not in hops:
            hops[w] = hops[v] + 1
            q.append(w)
print(hops["bridge"])
print(roads["home"]["bridge"], roads["home"]["lane"] + roads["lane"]["field"] + roads["field"]["bridge"])

In [ ]:
import heapq

def dijkstra(graph, start):
    dist = {start: 0}
    heap = [(0, start)]
    done = set()
    while heap:
        d, u = heapq.heappop(heap)
        if u in done:
            continue                       # an out-of-date entry
        done.add(u)
        for v, w in graph[u].items():
            nd = d + w
            if nd < dist.get(v, float("inf")):
                dist[v] = nd
                heapq.heappush(heap, (nd, v))
    return dist

roads = {"home": {"bridge": 9, "lane": 2}, "lane": {"field": 2}, "field": {"bridge": 2}, "bridge": {}}
print(dijkstra(roads, "home"))

In [ ]:
import heapq

def shortest_path(graph, start, goal):
    dist, parent = {start: 0}, {start: None}
    heap = [(0, start)]
    done = set()
    while heap:
        d, u = heapq.heappop(heap)
        if u in done:
            continue
        if u == goal:
            path = []
            while u is not None:
                path.append(u)
                u = parent[u]
            return d, path[::-1]
        done.add(u)
        for v, w in graph[u].items():
            if d + w < dist.get(v, float("inf")):
                dist[v], parent[v] = d + w, u
                heapq.heappush(heap, (d + w, v))
    return None

towns = {
    "Akure": {"Ondo": 50, "Ilesa": 95},
    "Ondo": {"Ife": 60, "Akure": 50},
    "Ilesa": {"Ibadan": 110, "Akure": 95},
    "Ife": {"Ibadan": 75, "Ilesa": 30},
    "Ibadan": {},
}
print(shortest_path(towns, "Akure", "Ibadan"), shortest_path(towns, "Ibadan", "Akure"))

In [ ]:
import heapq

def dijkstra(graph, start):
    dist, heap, done = {start: 0}, [(0, start)], set()
    while heap:
        d, u = heapq.heappop(heap)
        if u in done:
            continue
        done.add(u)
        for v, w in graph[u].items():
            if v not in done and d + w < dist.get(v, float("inf")):
                dist[v] = d + w
                heapq.heappush(heap, (d + w, v))
    return dist

g = {"A": {"B": 2, "C": 3}, "B": {"D": 1}, "C": {"B": -2}, "D": {}}
print(dijkstra(g, "A"), "but the true distances are B: 1, D: 2")

In [ ]:
import heapq

def widest(graph, start, goal):
    best = {start: float("inf")}
    heap = [(-float("inf"), start)]
    done = set()
    while heap:
        b, u = heapq.heappop(heap)
        b = -b
        if u in done:
            continue
        if u == goal:
            return b
        done.add(u)
        for v, cap in graph[u].items():
            nb = min(b, cap)
            if nb > best.get(v, 0):
                best[v] = nb
                heapq.heappush(heap, (-nb, v))
    return 0

bridges = {"depot": {"a": 4.2, "b": 3.1}, "a": {"site": 3.0}, "b": {"site": 4.0}, "site": {}}
print("tallest lorry:", widest(bridges, "depot", "site"))

> **Key idea.** In a weighted graph, the shortest path minimises the sum of weights, so BFS (which counts edges) doesn't work. Dijkstra's algorithm settles vertices in order of distance: pop the smallest tentative distance from a heap, skip out-of-date entries, and relax each outgoing edge, pushing a new entry when a distance improves. It's O((V + E) log V). Parent links rebuild paths; stopping when the target is popped saves work. It requires non-negative weights: a negative edge can improve a vertex already settled (Bellman-Ford handles those). The same pattern finds widest paths (combine with min, maximise), and searches over richer states (place plus whatever else the cost depends on).

### Exercise: Distances

Write `dijkstra(graph, start)` for a dict-of-dicts graph with non-negative weights: a dict from every vertex reachable from `start` to its distance, using `heapq` and lazy deletion. Then `farthest(graph, start)`: the reachable vertex with the greatest distance (alphabetically first on a tie) and its distance.

In [ ]:
import heapq

def dijkstra(graph, start):
    ___

def farthest(graph, start):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, itertools\ndef slow(graph, s):\n    dist = {s: 0}\n    changed = True\n    while changed:\n        changed = False\n        for u in list(dist):\n            for v, w in graph[u].items():\n                if dist[u] + w < dist.get(v, float(\"inf\")):\n                    dist[v] = dist[u] + w\n                    changed = True\n    return dist\ndef test_dist():\n    \"distances\"\n    roads = {\"home\": {\"bridge\": 9, \"lane\": 2}, \"lane\": {\"field\": 2}, \"field\": {\"bridge\": 2}, \"bridge\": {}}\n    assert dijkstra(roads, \"home\") == {\"home\": 0, \"lane\": 2, \"field\": 4, \"bridge\": 6}\n    assert dijkstra(roads, \"bridge\") == {\"bridge\": 0} and farthest(roads, \"home\") == (\"bridge\", 6)\n    rng = random.Random(1)\n    for _ in range(100):\n        n = rng.randint(1, 9)\n        g = {v: {w: rng.randint(0, 9) for w in range(n) if w != v and rng.random() < 0.35} for v in range(n)}\n        assert dijkstra(g, 0) == slow(g, 0)\ndef test_big():\n    \"fast\"\n    import time\n    n = 20_000\n    g = {i: {(i + 1) % n: 3, (i * 7 + 1) % n: 10} for i in range(n)}\n    t0 = time.perf_counter()\n    d = dijkstra(g, 0)\n    assert len(d) == n and time.perf_counter() - t0 < 4")

### Exercise: Shortest paths

Write `shortest_path(graph, start, goal)`: a pair `(distance, path)`, stopping as soon as the goal is settled, or `None` if it can't be reached. Then `path_tree(graph, start)`: a dict from each reachable vertex (other than `start`) to its parent in the shortest-path tree, and `route_to(tree, start, v)`: the path from start to v using it.

In [ ]:
import heapq

def shortest_path(graph, start, goal):
    ___

def path_tree(graph, start):
    ___

def route_to(tree, start, v):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\nTOWNS = {\n    \"Akure\": {\"Ondo\": 50, \"Ilesa\": 95},\n    \"Ondo\": {\"Ife\": 60, \"Akure\": 50},\n    \"Ilesa\": {\"Ibadan\": 110, \"Akure\": 95},\n    \"Ife\": {\"Ibadan\": 75, \"Ilesa\": 30},\n    \"Ibadan\": {},\n}\ndef test_paths():\n    \"paths\"\n    assert shortest_path(TOWNS, \"Akure\", \"Ibadan\") == (185, [\"Akure\", \"Ondo\", \"Ife\", \"Ibadan\"])\n    assert shortest_path(TOWNS, \"Ibadan\", \"Akure\") is None and shortest_path(TOWNS, \"Ife\", \"Ife\") == (0, [\"Ife\"])\n    tree = path_tree(TOWNS, \"Akure\")\n    assert tree[\"Ibadan\"] == \"Ife\" and tree[\"Ilesa\"] == \"Akure\" and route_to(tree, \"Akure\", \"Ibadan\") == [\"Akure\", \"Ondo\", \"Ife\", \"Ibadan\"]\n    rng = random.Random(2)\n    for _ in range(100):\n        n = rng.randint(1, 8)\n        g = {v: {w: rng.randint(1, 9) for w in range(n) if w != v and rng.random() < 0.4} for v in range(n)}\n        for t in range(n):\n            r = shortest_path(g, 0, t)\n            if r:\n                d, p = r\n                assert p[0] == 0 and p[-1] == t and sum(g[a][b] for a, b in zip(p, p[1:])) == d")

### Exercise: A message spreading

A message starts at one computer and travels along directed links, each taking a number of milliseconds. Write `time_for_all(links, n, start)`: the time until all n computers (numbered 1 to n) have the message, or -1 if some never get it; `links` is a list of `(from, to, ms)`. Then `last_to_hear(links, n, start)`: the sorted list of computers that get it last (`[]` if not everyone does).

In [ ]:
import heapq

def time_for_all(links, n, start):
    ___

def last_to_hear(links, n, start):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_delay():\n    \"spreading\"\n    links = [(2, 1, 1), (2, 3, 1), (3, 4, 1)]\n    assert time_for_all(links, 4, 2) == 2 and time_for_all([(1, 2, 1)], 2, 2) == -1 and time_for_all([], 1, 1) == 0\n    assert last_to_hear(links, 4, 2) == [4] and last_to_hear([(1, 2, 5), (1, 3, 5)], 3, 1) == [2, 3] and last_to_hear([(1, 2, 1)], 3, 1) == []")

### Exercise: The widest path

Each road has a height limit. Write `tallest_vehicle(graph, start, goal)`: the greatest height of vehicle that can get from start to goal (the largest, over all paths, of the smallest limit on the path), with the max-heap version of Dijkstra's algorithm; `0` if unreachable and infinity (`float("inf")`) if `start == goal`. Then `widest_route(graph, start, goal)`: one such route as a list, or `None`.

In [ ]:
import heapq

def tallest_vehicle(graph, start, goal):
    ___

def widest_route(graph, start, goal):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, itertools\ndef test_widest():\n    \"bottlenecks\"\n    bridges = {\"depot\": {\"a\": 4.2, \"b\": 3.1}, \"a\": {\"site\": 3.0}, \"b\": {\"site\": 4.0}, \"site\": {}}\n    assert tallest_vehicle(bridges, \"depot\", \"site\") == 3.1 and widest_route(bridges, \"depot\", \"site\") == [\"depot\", \"b\", \"site\"]\n    assert tallest_vehicle(bridges, \"site\", \"depot\") == 0 and widest_route(bridges, \"site\", \"depot\") is None\n    assert tallest_vehicle(bridges, \"a\", \"a\") == float(\"inf\")\n    rng = random.Random(3)\n    for _ in range(100):\n        n = rng.randint(2, 6)\n        g = {v: {w: rng.randint(1, 9) for w in range(n) if w != v and rng.random() < 0.5} for v in range(n)}\n        best = 0\n        for k in range(n - 1):\n            for mid in itertools.permutations(range(1, n - 1), k):\n                p = (0,) + mid + (n - 1,)\n                if all(b in g[a] for a, b in zip(p, p[1:])):\n                    best = max(best, min(g[a][b] for a, b in zip(p, p[1:])))\n        assert tallest_vehicle(g, 0, n - 1) == best\n        r = widest_route(g, 0, n - 1)\n        assert (r is None) == (best == 0) and (r is None or min(g[a][b] for a, b in zip(r, r[1:])) == best)")

### Exercise: One free ride

You have **one** coupon that makes a single road of your choice free. Write `cheapest_with_coupon(graph, start, goal)`: the least cost from start to goal using the coupon at most once (or `None`). Run Dijkstra on states `(place, used)`, where `used` is 0 or 1: from `(u, 0)` you may go to `(v, 0)` paying the weight, or to `(v, 1)` paying nothing; from `(u, 1)` only paying. Then `coupon_saving(graph, start, goal)`: how much the coupon saves compared with plain Dijkstra (`None` if unreachable).

In [ ]:
import heapq

def cheapest_with_coupon(graph, start, goal):
    ___

def coupon_saving(graph, start, goal):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, itertools\ndef test_coupon():\n    \"coupons\"\n    g = {\"A\": {\"B\": 10, \"C\": 1}, \"B\": {\"D\": 1}, \"C\": {\"D\": 30}, \"D\": {}}\n    assert cheapest_with_coupon(g, \"A\", \"D\") == 1 and coupon_saving(g, \"A\", \"D\") == 10\n    assert cheapest_with_coupon(g, \"D\", \"A\") is None and coupon_saving(g, \"D\", \"A\") is None and cheapest_with_coupon(g, \"A\", \"A\") == 0\n    rng = random.Random(4)\n    for _ in range(100):\n        n = rng.randint(2, 6)\n        g = {v: {w: rng.randint(1, 20) for w in range(n) if w != v and rng.random() < 0.5} for v in range(n)}\n        best = None\n        for k in range(n - 1):\n            for mid in itertools.permutations(range(1, n - 1), k):\n                p = (0,) + mid + (n - 1,)\n                if all(b in g[a] for a, b in zip(p, p[1:])):\n                    ws = [g[a][b] for a, b in zip(p, p[1:])]\n                    c = sum(ws) - max(ws)\n                    best = c if best is None else min(best, c)\n        assert cheapest_with_coupon(g, 0, n - 1) == best")

### Bug hunt: Settled too soon

This version marks a vertex as done when it's **pushed**, as BFS does, so a vertex reached first by a long route can never be improved: this prints `9` for the bridge. Settle vertices when they're **popped** so it prints `6`.

In [ ]:
import heapq

roads = {"home": {"bridge": 9, "lane": 2}, "lane": {"field": 2}, "field": {"bridge": 2}, "bridge": {}}
dist, heap, done = {"home": 0}, [(0, "home")], {"home"}
while heap:
    d, u = heapq.heappop(heap)
    for v, w in roads[u].items():
        if v not in done:
            done.add(v)
            dist[v] = d + w
            heapq.heappush(heap, (d + w, v))
print(dist["bridge"])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"settle when popped\"\n    expect_output(\"6\")")

### Bug hunt: Counting roads, not minutes

This relaxes edges with `d + 1` instead of `d + w`, so it finds the route with the fewest roads, like BFS: this prints `9`. Add the edge's weight so it prints `6`.

In [ ]:
import heapq

roads = {"home": {"bridge": 9, "lane": 2}, "lane": {"field": 2}, "field": {"bridge": 2}, "bridge": {}}
dist, heap, done, parent = {"home": 0}, [(0, "home")], set(), {"home": None}
while heap:
    d, u = heapq.heappop(heap)
    if u in done:
        continue
    done.add(u)
    for v, w in roads[u].items():
        if d + 1 < dist.get(v, float("inf")):
            dist[v], parent[v] = d + 1, u
            heapq.heappush(heap, (d + 1, v))
v, total = "bridge", 0
while parent[v] is not None:
    total += roads[parent[v]][v]
    v = parent[v]
print(total)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"weights\"\n    expect_output(\"6\")")

### Bug hunt: Vertices that can't be compared

The vertices here are dicts describing places, and when two heap entries have the same distance, `heapq` compares the dicts and crashes with a `TypeError`. Put a counter between distance and vertex in each entry so it prints `3`.

In [ ]:
import heapq

a, b, c, d = {"name": "a"}, {"name": "b"}, {"name": "c"}, {"name": "d"}
graph = {"a": [(b, 1), (c, 1)], "b": [(d, 2)], "c": [(d, 2)], "d": []}
dist, heap, done = {"a": 0}, [(0, a)], set()
while heap:
    du, u = heapq.heappop(heap)
    if u["name"] in done:
        continue
    done.add(u["name"])
    for v, w in graph[u["name"]]:
        if du + w < dist.get(v["name"], float("inf")):
            dist[v["name"]] = du + w
            heapq.heappush(heap, (du + w, v))
print(dist["d"])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a tie-breaker\"\n    expect_output(\"3\")")

### Puzzle: Dijkstra's algorithm

Arrange the lines into Dijkstra's algorithm with lazy deletion. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
# Directed: S->A 4, S->B 1, B->A 2, A->C 1, B->C 5.

p = ___   # the order in which Dijkstra settles the vertices, from S
q = ___   # the distance to C
r = ___   # the shortest path to C, as a list
s = ___   # how many times A's tentative distance is improved after it is first set
t = ___   # True or False: Dijkstra's algorithm is correct with negative edge weights

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == [\"S\", \"B\", \"A\", \"C\"] and q == 4 and r == [\"S\", \"B\", \"A\", \"C\"], \"p, q and r\"\n    assert s == 1 and t is False, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: minutes and changes

Last week's transport planner counted stops or changes, but travellers care about **minutes**, and changing lines takes time. Lines are given as dicts `{"line": name, "stops": [...], "minutes": [...]}`, where `minutes[i]` is the time between `stops[i]` and `stops[i + 1]` (both directions). Changing from one line to another at a station costs a fixed `change` number of minutes; starting on any line at the start station is free.

Write `quickest(lines, change, start, goal)`: `(minutes, plan)`, where `plan` is a list of `(line, from_station, to_station)` legs, one per line ridden, or `None` if unreachable (`(0, [])` if `start == goal`). Run Dijkstra on states `(station, line)`: riding to the next station on the same line costs its minutes; switching to another line at the same station costs `change`. When two plans take equally long, either is fine, but the checks compare the total with a slow method on random networks.

In [ ]:
import heapq

def quickest(lines, change, start, goal):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, heapq\nLINES = [\n    {\"line\": \"Red\", \"stops\": [\"Harbour\", \"Market\", \"Cathedral\", \"University\"], \"minutes\": [3, 4, 6]},\n    {\"line\": \"Blue\", \"stops\": [\"Airport\", \"Station\", \"Cathedral\"], \"minutes\": [12, 5]},\n    {\"line\": \"Green\", \"stops\": [\"Station\", \"Market\"], \"minutes\": [2]},\n]\ndef slow(lines, change, start, goal):\n    # Bellman-Ford style relaxation on (station, line) states\n    edges = []\n    states = set()\n    for L in lines:\n        for i, (a, b) in enumerate(zip(L[\"stops\"], L[\"stops\"][1:])):\n            edges += [((a, L[\"line\"]), (b, L[\"line\"]), L[\"minutes\"][i]), ((b, L[\"line\"]), (a, L[\"line\"]), L[\"minutes\"][i])]\n            states |= {(a, L[\"line\"]), (b, L[\"line\"])}\n    for s in list(states):\n        for t in list(states):\n            if s[0] == t[0] and s[1] != t[1]:\n                edges.append((s, t, change))\n    dist = {s: 0 for s in states if s[0] == start}\n    for _ in range(len(states)):\n        for a, b, w in edges:\n            if a in dist and dist[a] + w < dist.get(b, float(\"inf\")):\n                dist[b] = dist[a] + w\n    best = [d for s, d in dist.items() if s[0] == goal]\n    return min(best) if best else None\ndef test_example():\n    \"an example\"\n    assert quickest(LINES, 5, \"Harbour\", \"Airport\") == (27, [(\"Red\", \"Harbour\", \"Market\"), (\"Green\", \"Market\", \"Station\"), (\"Blue\", \"Station\", \"Airport\")])\n    assert quickest(LINES, 5, \"Harbour\", \"University\") == (13, [(\"Red\", \"Harbour\", \"University\")])\n    assert quickest(LINES, 100, \"Harbour\", \"Airport\")[0] == 3 + 4 + 100 + 5 + 12\n    assert quickest(LINES, 5, \"Market\", \"Market\") == (0, []) and quickest(LINES + [{\"line\": \"Grey\", \"stops\": [\"Mill\", \"Quarry\"], \"minutes\": [4]}], 5, \"Mill\", \"Harbour\") is None\ndef test_random():\n    \"against a slow method\"\n    rng = random.Random(5)\n    names = [f\"s{i}\" for i in range(8)]\n    for _ in range(80):\n        lines = []\n        for k in range(rng.randint(1, 4)):\n            stops = rng.sample(names, rng.randint(2, 4))\n            lines.append({\"line\": f\"L{k}\", \"stops\": stops, \"minutes\": [rng.randint(1, 9) for _ in stops[1:]]})\n        change = rng.randint(0, 6)\n        here = [s for L in lines for s in L[\"stops\"]]\n        a, b = rng.choice(here), rng.choice(here)\n        got = quickest(lines, change, a, b)\n        want = 0 if a == b else slow(lines, change, a, b)\n        assert (got is None and want is None) or got[0] == want, (lines, change, a, b, got, want)\n        if got and a != b:\n            plan = got[1]\n            assert plan[0][1] == a and plan[-1][2] == b")

## Warm-up

BFS says the bridge is one road away, and it is, but that road is 9 minutes long. Going round by the lane and the field takes three roads and only 6 minutes. With weights, "fewest edges" and "shortest" are different questions.

## Why breadth-first isn't enough

A **weighted graph** gives each edge a number: its length, cost or time. The length of a path is the **sum** of its weights, and the shortest path is the one with the smallest sum. Store it as a dict of dicts (Week 50 Day 1): `graph[u][v]` is the weight of the edge u → v.

BFS settles vertices in order of number of edges, which is right only when every edge weighs the same. One way to rescue it is to split every edge of weight w into w edges of weight 1, but weights can be huge (or not whole numbers), so that's hopeless in general. What's needed is a search that settles vertices in order of **distance**, not of edge count.

## Settling the closest

Dijkstra's algorithm keeps a **tentative distance** for each vertex: the shortest way found so far (infinity at first, 0 for the start). Then, repeatedly:

1. Take the vertex u with the **smallest** tentative distance among those not yet settled, and **settle** it: its distance is now final.
2. **Relax** each edge u → v: if `dist[u] + weight(u, v)` is less than `dist[v]`, lower `dist[v]` to it (and remember u as v's parent).

Why is the smallest tentative distance final? Any other route to u would have to leave the settled region through some unsettled vertex x, and already `dist[x] >= dist[u]`; with **no negative weights**, the rest of that route can only add more. So nothing can beat `dist[u]`.

A heap finds the smallest quickly. Python's `heapq` can't lower a value that's already in the heap, so use **lazy deletion** (Week 47 Day 2): push a new `(distance, vertex)` entry whenever a distance improves, and when popping, skip entries whose distance is out of date. The heap may hold up to E entries, so the whole algorithm is **O((V + E) log V)** (log E and log V differ by at most a factor of 2).

The entries are tuples `(distance, vertex)`, so equal distances compare vertices next; that's fine when vertices are strings or numbers. For vertices that can't be compared, put a counter between them (Week 47 Day 2).

## Paths and early stopping

Recording each vertex's **parent** whenever its distance improves rebuilds the path, exactly as with BFS: walk back from the target and reverse. The parent links form a **shortest-path tree**.

If you only need the distance to one target, you can **stop** as soon as the target is popped (settled): its distance is final, and everything still in the heap is at least as far. On a large map, that can save most of the work. (Next Friday's A* search goes further, steering the search towards the target.)

## Negative weights

Dijkstra's algorithm settles a vertex for good. A **negative** edge can make a later route shorter than one already settled, and the algorithm never looks back:

```text
        A ---2---> B ---1---> D
        |          ^
        3          | -2
        v          |
        C ---------+
   true shortest to B: A -> C -> B = 1, and to D: 2.  Dijkstra settles B at 2 first.
```

Negative weights are real (a refund, a gain in energy, an exchange rate taken as a logarithm), so they need a different algorithm: tomorrow's Bellman-Ford. And with a **negative cycle** (a loop whose weights sum below zero), "shortest" stops making sense: going round again is always shorter.

## Changing the question

Dijkstra's algorithm works for any way of combining weights along a path where adding an edge never makes a path **better** and paths are compared by a total order. Two common adaptations:

- **Widest (bottleneck) path**: a path's capacity is its **smallest** edge (a lorry's height limit along a route, a network's bandwidth). Find the path with the largest capacity: keep the **largest** bottleneck so far, combine with `min`, and use a max-heap (negate).
- **Richer states**: when the cost depends on more than where you are (which line you're on, whether you've used a discount, how much fuel is left), make the **vertex** a tuple of everything that matters, `(place, extra)`, and run Dijkstra on that bigger implicit graph. Week 50's "the right graph for the question" again.

## Practice

## Quiz

## Daily challenge

## Before you go

Look up the journey times between four places you know and draw them as a weighted graph. Run Dijkstra's algorithm by hand from one of them, writing down the tentative distances after each step. Tomorrow: Bellman-Ford, which copes with negative weights and spots negative cycles.